# ASR Shootout — Colab / Kaggle Runner

**Models:** Deepgram nova-2 (baseline) · faster-whisper large-v3 · AI4Bharat IndicWav2Vec

**Runtime required:** GPU → T4
- Colab: Runtime → Change runtime type → T4 GPU
- Kaggle: Accelerator → GPU T4 x2

Run each cell top-to-bottom.

## Step 1 — Mount Drive and auto-detect project root

In [ ]:
import os, sys
from pathlib import Path

# Step back to /content first in case the previous Drive mount dropped
os.chdir('/content')

# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Hardcoded path — asr-benchmark is inside MyDrive/colab/
PROJECT_ROOT = '/content/drive/MyDrive/colab/asr-benchmark'

if not Path(PROJECT_ROOT).exists():
    raise FileNotFoundError(f'Not found: {PROJECT_ROOT}  — check the folder location in Drive')

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print('Working directory:', os.getcwd())
print('Audio files:', len(list((Path(PROJECT_ROOT) / 'audio' / 'raw').glob('*.wav'))))

## Step 2 — Install dependencies

In [ ]:
!pip install -q \
    "deepgram-sdk>=3.0.0,<4.0.0" \
    faster-whisper>=1.0.0 \
    jiwer>=3.0.0 \
    rapidfuzz>=3.0.0 \
    pandas>=2.0.0 \
    matplotlib>=3.7.0 \
    transformers>=4.35.0 \
    soundfile>=0.12.0 \
    librosa>=0.10.0 \
    pyctcdecode>=0.5.0
print('Done.')

## Step 3 — Set API keys

AI4Bharat uses HuggingFace local inference — **no API key needed**.

Only Deepgram needs a key.

In [ ]:
import os

# Option A — Colab Secrets (Key icon in left sidebar → add DEEPGRAM_API_KEY)
try:
    from google.colab import userdata
    os.environ['DEEPGRAM_API_KEY'] = userdata.get('DEEPGRAM_API_KEY')
    print('Deepgram key loaded from Colab Secrets.')
except Exception:
    # Option B — paste key directly (do not share this notebook with the key in it)
    os.environ['DEEPGRAM_API_KEY'] = 'YOUR_DEEPGRAM_KEY_HERE'
    print('Deepgram key set directly (Option B).')

# Use medium for quick dev iteration; large-v3 for final benchmark run
os.environ['WHISPER_MODEL_SIZE'] = 'large-v3'

## Step 4 — Verify GPU and files

In [ ]:
import subprocess, json
from pathlib import Path

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
print('GPU:', result.stdout.strip() or 'NOT DETECTED — switch runtime to T4')

audio_dir = Path(PROJECT_ROOT) / 'audio' / 'raw'
wavs = sorted(audio_dir.glob('*.wav'))
print(f'\nAudio files: {len(wavs)}')
for w in wavs:
    print(f'  {w.name}')

gt_path = Path(PROJECT_ROOT) / 'ground_truth' / 'ground_truth.json'
with gt_path.open() as f:
    gt = json.load(f)
empty = [k for k, v in gt['samples'].items() if not v.get('transcript','').strip()]
print(f'\nGround truth entries: {len(gt["samples"])}  (empty: {len(empty)})')
if empty:
    print('WARNING — fill these before running pipeline:', empty)
else:
    print('All transcripts filled. Ready to run.')

## Step 5 — Run pipeline

Expected time: ~15–25 min total (Whisper downloads ~3GB on first run).

To run one model at a time: `--models deepgram` / `--models whisper` / `--models ai4bharat`

In [ ]:
!python scripts/pipeline.py --models deepgram whisper ai4bharat

## Step 6 — Run analysis and generate charts

In [ ]:
!python scripts/analyze.py

## Step 7 — View charts inline

In [ ]:
from IPython.display import Image, display
from pathlib import Path

chart_dir = Path(PROJECT_ROOT) / 'results' / 'aggregated'
for png in sorted(chart_dir.glob('*.png')):
    print(f'\n--- {png.name} ---')
    display(Image(str(png)))

## Step 8 — Inspect failure cases

In [ ]:
import pandas as pd, subprocess, os
from pathlib import Path

failures_csv = Path(PROJECT_ROOT) / 'results' / 'aggregated' / 'failure_cases.csv'

# Check what raw result CSVs exist first
raw_dir = Path(PROJECT_ROOT) / 'results' / 'raw'
raw_csvs = list(raw_dir.glob('*.csv')) if raw_dir.exists() else []
print(f"Raw CSVs found: {[f.name for f in raw_csvs]}")

if not raw_csvs:
    print("\nNo pipeline results found. Run cell 5 (pipeline.py) first.")
else:
    # Run analyze.py and show full output + errors
    result = subprocess.run(
        ['python', 'scripts/analyze.py'],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print("=== ERROR ===")
        print(result.stderr)
    else:
        if failures_csv.exists():
            fails = pd.read_csv(failures_csv)
            if fails.empty:
                print('No failure cases — all locality names were correctly identified.')
            else:
                display(fails[['model','filename','condition','ground_truth','transcript','wer','fuzzy_score']].head(20))